# Investigating question responses which indicate cognitive dissonance

In [ ]:
from pathlib import Path

import polars as pl
import matplotlib.pyplot as plt
import seaborn as sns


from climate_attitudes.settings import Config
from climate_attitudes.dataset import Dataset
from climate_attitudes import configure_mpl

FONT_PATH = Path("../fonts")
configure_mpl(FONT_PATH)

plt.rc("figure", dpi=150)

In [ ]:
config = Config(_env_file="../.env")
dataset = Dataset.load(config).transform(
    pl.col(r"^cc(4|5)_(world|wealthUS|poorUS|comm)$").replace({1: 0, 2: 1, 99: 2}),
    pl.col("cc1").replace({1: 2, 99: 1}),
)
resp = dataset.response.collect()

## Future climate change impacts, but not current

In [ ]:
plot_data = (
    resp.filter(pl.all_horizontal(pl.col("cc4_world", "cc5_world").is_not_null()))
    .group_by("cc4_world", "cc5_world")
    .agg(pl.len())
    .sort(by=("cc5_world", "cc4_world"))
    .pivot("cc4_world", index="cc5_world")
    .select(pl.col(r"^\d$"))
    .to_numpy()[::-1]
)

plot_data = plot_data / plot_data.sum(axis=0)

fig, ax = plt.subplots(constrained_layout=True)
sns.heatmap(
    plot_data,
    xticklabels=[
        "Not at all",
        "Only a little",
        "Don't know",
        "Moderate",
        "A great deal",
    ],
    yticklabels=list(
        reversed(
            ["Not at all", "Only a little", "Don't know", "Moderate", "A great deal"]
        )
    ),
    ax=ax,
)
ax.set_xlabel("Current CC Impacts (world)")
ax.set_ylabel("Future CC Impacts (world)");

In [ ]:
plot_data = (
    resp.filter(pl.all_horizontal(pl.col("cc4_comm", "cc5_comm").is_not_null()))
    .group_by("cc4_comm", "cc5_comm")
    .agg(pl.len())
    .sort(by=("cc5_comm", "cc4_comm"))
    .pivot("cc4_comm", index="cc5_comm")
    .select(pl.col(r"^\d$"))
    .to_numpy()[::-1]
)

plot_data = plot_data / plot_data.sum(axis=0)

fig, ax = plt.subplots(constrained_layout=True)
sns.heatmap(
    plot_data,
    xticklabels=[
        "Not at all",
        "Only a little",
        "Don't know",
        "Moderate",
        "A great deal",
    ],
    yticklabels=list(
        reversed(
            ["Not at all", "Only a little", "Don't know", "Moderate", "A great deal"]
        )
    ),
    ax=ax,
)
ax.set_xlabel("Current CC Impacts (community)")
ax.set_ylabel("Future CC Impacts (community)");

## Future CC impacts, but not in own community

In [ ]:
plot_data = (
    resp.filter(pl.all_horizontal(pl.col("cc5_world", "cc5_comm").is_not_null()))
    .group_by("cc5_world", "cc5_comm")
    .agg(pl.len())
    .sort(by=("cc5_comm", "cc5_world"))
    .pivot("cc5_world", index="cc5_comm")
    .select(pl.col(r"^\d$"))
    .to_numpy()[::-1]
)

plot_data = plot_data / plot_data.sum(axis=0)

fig, ax = plt.subplots(constrained_layout=True)
sns.heatmap(
    plot_data,
    xticklabels=[
        "Not at all",
        "Only a little",
        "Don't know",
        "Moderate",
        "A great deal",
    ],
    yticklabels=list(
        reversed(
            ["Not at all", "Only a little", "Don't know", "Moderate", "A great deal"]
        )
    ),
    ax=ax,
)
ax.set_xlabel("Future CC Impacts (world)")
ax.set_ylabel("Future CC Impacts (community)");

## Concern about extreme weather, but limited preparation

In [ ]:
fig, ax = plt.subplots(constrained_layout=True)
sns.heatmap(
    resp.group_by("ew5", "ew6")
    .agg(pl.len())
    .select("ew5", "ew6", (pl.col("len") / pl.col("len").sum()).alias("prop"))
    .sort(by=("ew5", "ew6"))
    .pivot("ew6", index="ew5")
    .select(pl.col(r"^\d$"))
    .to_numpy()[::-1],
    xticklabels=["Not at all", "Only a little", "A moderate amount", "A great deal"],
    yticklabels=list(
        reversed(["Not at all", "Only a little", "A moderate amount", "A great deal"])
    ),
    ax=ax,
)
ax.set_xlabel("Concern about future extreme weather")
ax.set_ylabel("Preparation for future extreme weather");

In [ ]:
resp = resp.with_columns(
    ((pl.col("ew5") >= 3) & (pl.col("ew6") <= 1)).alias("ew_underprepared")
)

In [ ]:
sns.displot(
    resp.filter(pl.col("ew5") >= 3).with_columns(pl.col("ew1").list.len()).to_pandas(),
    x="ew1",
    hue="ew_underprepared",
    stat="probability",
    discrete=True,
    shrink=0.8,
    multiple="dodge",
    common_norm=False,
)

In [ ]:
sns.displot(
    resp.filter(pl.col("ew5") >= 3).to_pandas(),
    x="dem_age",
    kind="kde",
    hue="ew_underprepared",
    multiple="layer",
    common_norm=False,
)

## Concern about extreme weather, but doesn't believe in climate change

In [ ]:
fig, ax = plt.subplots(constrained_layout=True)
sns.heatmap(
    resp.group_by("ew5", "cc1")
    .agg(pl.len())
    .select("ew5", "cc1", (pl.col("len") / pl.col("len").sum()).alias("prop"))
    .sort(by=("ew5", "cc1"))
    .pivot("cc1", index="ew5")
    .select(pl.col(r"^\d$"))
    .to_numpy()[::-1],
    xticklabels=["No", "Don't know", "Yes"],
    yticklabels=list(
        reversed(["Not at all", "Only a little", "A moderate amount", "A great deal"])
    ),
    ax=ax,
)
ax.set_xlabel("Concern about future extreme weather")
ax.set_ylabel("Preparation for future extreme weather");

In [ ]:
sns.displot(
    resp.filter(pl.col("cc1") == 0, pl.col("ew5") >= 3).to_pandas(), x="pol_affiliation"
)

## Believes in climate change but doesn't think it is a problem

In [ ]:
fig, ax = plt.subplots(constrained_layout=True)
plot_data = (
    resp.filter(pl.all_horizontal(pl.col("cc1", "cc3").is_not_null()))
    .group_by("cc1", "cc3")
    .agg(pl.len())
    # .select(
    #     "cc1", "cc3",
    #     (pl.col("len") / pl.col("len").sum()).alias("prop")
    # )
    .sort(by=("cc3", "cc1"))
    .pivot("cc1", index="cc3")
    .select(pl.col(r"^\d$"))
    .to_numpy()[::-1]
)
plot_data = plot_data / plot_data.sum(axis=0)
sns.heatmap(
    plot_data,
    xticklabels=["No", "Don't know", "Yes"],
    yticklabels=list(
        reversed(
            [
                "Not a problem at all",
                "A minor problem",
                "A serious problem but not a crisis",
                "A significant crisis",
            ]
        )
    ),
    ax=ax,
)
ax.set_xlabel("Belief in climate change")
ax.set_ylabel("Climate change crisis level");

## Believes others _should_ change behaviour but also that they won't

In [ ]:
plot_data = (
    resp.filter(pl.all_horizontal(pl.col("cvcc4_should", "cvcc4_will").is_not_null()))
    .group_by("cvcc4_should", "cvcc4_will")
    .agg(pl.len())
    .sort(by=("cvcc4_will", "cvcc4_should"))
    .pivot("cvcc4_should", index="cvcc4_will")
    .select(pl.col(r"^\d$"))
    .to_numpy()[::-1]
)

plot_data = plot_data / plot_data.sum(axis=0)

fig, ax = plt.subplots(constrained_layout=True)
sns.heatmap(
    plot_data,
    xticklabels=[
        "Strongly disagree",
        "Disagree",
        "Neither agree nor disagree",
        "Agree",
        "Strongly agree",
    ],
    yticklabels=list(
        reversed(
            [
                "Strongly disagree",
                "Disagree",
                "Neither agree not disagree",
                "Agree",
                "Strongly agree",
            ]
        )
    ),
    ax=ax,
)
ax.set_xlabel("People should change behaviour")
ax.set_ylabel("People will change behaviour");